# MCL Watermarking Results Analysis

This notebook provides comprehensive visualizations of the Markov Chain Lock (MCL) watermarking experiment results using **Plotly** for interactive plots.

**Experiment Overview:**
- **Model:** Llama-3.2-3B-Instruct
- **Dataset:** 173 curated Wikipedia concepts
- **Configurations:** 28 (7 state counts × 4 overlap ratios)

---

## 1. Setup and Data Loading

In [70]:
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# Set plot output directory
PICTURES_DIR = '../docs/pictures'
os.makedirs(PICTURES_DIR, exist_ok=True)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [71]:
# Load the detailed comparison data
DATA_DIR = '../data/curated_wiki_dataset_20260201_112721'

with open(f'{DATA_DIR}/detailed_comparison.json', 'r') as f:
    detailed_data = json.load(f)

with open(f'{DATA_DIR}/summary.json', 'r') as f:
    summary_data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(detailed_data)

# Add overlap percentage column for better labeling
df['overlap_pct'] = (df['overlap'] * 100).astype(int).astype(str) + '%'

print(f"Loaded {len(df)} configurations")
df.head()

Loaded 28 configurations


,config,num_states,overlap,non_wm_avg,non_wm_std,non_wm_min,non_wm_max,non_wm_fpr,wm_avg,wm_std,wm_min,wm_max,wm_tpr,wm_ppl_avg,gap,baseline,overlap_pct
0,states2_overlap0pct,2,0.00,1.000000,0.000000,1.000000,1.000000,1.00000,1.000000,0.000000,1.000000,1.000000,1.0,1.286345,0.000000,1.0,0%
1,states2_overlap5pct,2,0.05,1.000000,0.000000,1.000000,1.000000,1.00000,1.000000,0.000000,1.000000,1.000000,1.0,1.286345,0.000000,1.0,5%
2,states2_overlap10pct,2,0.10,1.000000,0.000000,1.000000,1.000000,1.00000,1.000000,0.000000,1.000000,1.000000,1.0,1.286345,0.000000,1.0,10%
3,states2_overlap15pct,2,0.15,1.000000,0.000000,1.000000,1.000000,1.00000,1.000000,0.000000,1.000000,1.000000,1.0,1.286345,0.000000,1.0,15%
4,states4_overlap0pct,4,0.00,0.489017,0.045152,0.333333,0.606667,0.33526,0.988722,0.016865,0.871622,0.993333,1.0,4.891848,0.499704,0.5,0%


---

## 2. Key Observations

Our experiments across **28 configurations** (7 state counts × 4 overlap ratios) reveal the following findings:

---

### 2.1 Perfect Separability for S ≥ 5 with ρ = 0%

For configurations with **5 or more states** and **0% vocabulary overlap**, we observe *perfect separability* between watermarked and non-watermarked text: the **minimum** watermarked score exceeds the **maximum** non-watermarked score.

**Why this matters:** This enables **zero-error detection** — any text with φ(t) > τ = 0.5 is definitively watermarked, with 0% false positives and 100% true positives. The Markov chain's state rotation creates a sufficiently distinct signal that random text cannot accidentally replicate.

---

### 2.2 Optimal Configuration: S = 7 States, ρ = 0%

Among all perfect-detection configurations, **S = 7 with 0% overlap** achieves the best quality-detection trade-off:
- **Detection Rate:** 100% TPR, 0% FPR
- **Text Quality:** Lowest perplexity (PPL ≈ 4.20) among perfect configs
- **Score Gap:** 0.82 (strong separation)

**Why this matters:** Fewer states (S=5) achieve perfect detection but with higher perplexity (PPL ≈ 6.8). More states (S≥9) offer no detection improvement but increase computational overhead. S = 7 is the **Pareto-optimal** choice.

---

### 2.3 Binary Alternation (S = 2) Fails Completely

With only **2 states**, both watermarked and non-watermarked text achieve a fingerprint score of **φ(t) = 1.0**, making detection impossible.

**Why this happens:** Binary alternation is too simple — natural text frequently contains alternating token patterns that match the trivial rotation. The vocabulary partitions are too coarse to create discriminative signals. This validates our theoretical lower bound requiring S ≥ 4 for meaningful detection.

---

### 2.4 Four States (S = 4) is Borderline

With **S = 4** and 0% overlap:
- **True Positive Rate:** 100% (all watermarked text detected)
- **False Positive Rate:** 33.5% (1 in 3 non-WM texts falsely detected)

**Why this happens:** Four states provide marginal separation, but the expected non-watermarked score (≈ 0.49) is just below the τ = 0.5 threshold. Statistical variation causes some non-watermarked texts to exceed the threshold. For reliable detection, S ≥ 5 is required.

---

### 2.5 Overlap Degrades Detection Severely

Even small vocabulary overlap **destroys** watermark effectiveness:

| Overlap (ρ) | TPR for S = 7 | Score Gap |
|-------------|---------------|-----------|
| 0% | 100% | 0.82 |
| 5% | 92% | 0.35 |
| 10% | 60% | 0.21 |
| 15% | 12.7% | 0.14 |

**Why this happens:** Overlap allows tokens to appear in multiple state partitions, weakening the Markov chain lock. The fingerprint score becomes noisy because the same token can match multiple states, reducing the signal-to-noise ratio of the watermark.

---


## 3. Score Analysis

Combined view of watermarked vs non-watermarked scores and false positive rates for 0% overlap configurations.

In [72]:
# Figure 1: Score Comparison & FPR Analysis (2x1)
from plotly.subplots import make_subplots

df_0pct = df[df['overlap'] == 0.0].copy()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Watermarked vs Non-Watermarked Scores</b>', '<b>False Positive Rate by State Count</b>'),
    horizontal_spacing=0.12
)

# Left: Score Comparison
fig.add_trace(go.Bar(name='Watermarked', x=df_0pct['num_states'].astype(str), y=df_0pct['wm_avg'],
                     marker_color='#2ecc71', text=df_0pct['wm_avg'].round(3), textposition='outside'), row=1, col=1)
fig.add_trace(go.Bar(name='Non-Watermarked', x=df_0pct['num_states'].astype(str), y=df_0pct['non_wm_avg'],
                     marker_color='#e74c3c', text=df_0pct['non_wm_avg'].round(3), textposition='outside'), row=1, col=1)
fig.add_hline(y=0.5, line_dash="dash", line_color="gray", row=1, col=1)
fig.add_annotation(text="τ=0.5", x=6.5, y=0.55, showarrow=False, font=dict(size=10), row=1, col=1)

# Right: FPR
colors = ['#e74c3c' if fpr > 0 else '#2ecc71' for fpr in df_0pct['non_wm_fpr']]
fig.add_trace(go.Bar(x=df_0pct['num_states'].astype(str), y=df_0pct['non_wm_fpr']*100, marker_color=colors,
                     text=[f"{fpr*100:.1f}%" for fpr in df_0pct['non_wm_fpr']], textposition='outside',
                     showlegend=False), row=1, col=2)

fig.update_xaxes(title_text='Number of States (S)', row=1, col=1)
fig.update_xaxes(title_text='Number of States (S)', row=1, col=2)
fig.update_yaxes(title_text='Average Fingerprint Score φ(t)', row=1, col=1)
fig.update_yaxes(title_text='False Positive Rate (%)', row=1, col=2)

fig.update_layout(
    title=dict(text='<b>Figure 1: Score & Detection Metrics (0% Overlap)</b><br><sup>S≥5 achieves 100% detection with 0% false positives</sup>', x=0.5),
    template='plotly_white', barmode='group', height=450, width=1100,
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.25),
    margin=dict(b=80)
)

fig.write_image(f'{PICTURES_DIR}/fig1_score_fpr.png', scale=2)
fig.show()


## 4. Overlap Impact Analysis

How vocabulary overlap affects score gap and detection rate across all configurations.

In [73]:
# Figure 2: Overlap Impact Analysis (side-by-side with proper spacing)
from plotly.subplots import make_subplots

# Create 1x2 horizontal layout with more spacing
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Score Gap Heatmap</b>', '<b>Detection Rate Degradation</b>'),
    horizontal_spacing=0.22,  # Increased spacing for colorbar
    column_widths=[0.4, 0.6]  # Give more room to the line chart
)

# Left: Heatmap
pivot_gap = df.pivot(index='num_states', columns='overlap', values='gap')
pivot_gap.columns = ['0%', '5%', '10%', '15%']

fig.add_trace(go.Heatmap(
    z=pivot_gap.values, x=['0%', '5%', '10%', '15%'], y=pivot_gap.index.tolist(),
    colorscale='RdYlGn', text=np.round(pivot_gap.values, 2), texttemplate='%{text:.2f}',
    colorbar=dict(title='Gap', x=0.38, len=0.9, thickness=15),
    showscale=True
), row=1, col=1)

# Right: Detection Rate Lines
df_valid = df[df['num_states'] >= 4].copy()
colors = px.colors.qualitative.Set2
for i, s in enumerate([4, 5, 7, 9, 11, 15]):
    subset = df_valid[df_valid['num_states'] == s]
    fig.add_trace(go.Scatter(
        x=subset['overlap_pct'], y=subset['wm_tpr'], mode='lines+markers',
        name=f'S={s}', line=dict(color=colors[i], width=2), marker=dict(size=10)
    ), row=1, col=2)

fig.update_xaxes(title_text='Overlap Ratio (ρ)', row=1, col=1)
fig.update_xaxes(title_text='Overlap Ratio (ρ)', row=1, col=2)
fig.update_yaxes(title_text='Number of States (S)', row=1, col=1)
fig.update_yaxes(title_text='Detection Rate (TPR)', tickformat=',.0%', range=[0, 1.08], row=1, col=2)

fig.update_layout(
    title=dict(text='<b>Figure 2: Overlap Impact on Watermark Performance</b><br><sup>Higher overlap severely degrades both score gap and detection rate</sup>', x=0.5),
    template='plotly_white', height=450, width=1200,
    legend=dict(title='States', orientation='h', yanchor='top', y=-0.15, xanchor='center', x=0.7),
    margin=dict(b=80, l=60, r=40)
)

fig.write_image(f'{PICTURES_DIR}/fig2_overlap_impact.png', scale=2)
fig.show()


## 5. Quality-Detection Trade-off

Comparing perplexity (text quality) vs detection rate across configurations.

In [74]:
# Create 2x1 subplot for cleaner PPL vs Detection Rate trade-off
from plotly.subplots import make_subplots

df_plot = df[df['num_states'] >= 4].copy()
df_plot['config_label'] = 'S=' + df_plot['num_states'].astype(str) + ', ρ=' + df_plot['overlap_pct']

# Split data: 0% overlap vs higher overlap
df_optimal = df_plot[df_plot['overlap'] == 0.0].copy()
df_degraded = df_plot[df_plot['overlap'] > 0.0].copy()

# Create 2x1 subplots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Optimal (0% Overlap)</b>', '<b>With Overlap (5-15%)</b>'),
    horizontal_spacing=0.12
)

# Color scale for states
colors = px.colors.qualitative.Set1
state_colors = {4: colors[0], 5: colors[1], 7: colors[2], 9: colors[3], 11: colors[4], 15: colors[5]}

# Left panel: 0% overlap configurations
for i, (_, row) in enumerate(df_optimal.iterrows()):
    s = int(row['num_states'])
    fig.add_trace(
        go.Scatter(
            x=[row['wm_ppl_avg']],
            y=[row['wm_tpr']],
            mode='markers+text',
            marker=dict(size=20 + row['gap']*15, color=state_colors[s], line=dict(width=2, color='white')),
            text=[f"S={s}"],
            textposition='middle center',
            textfont=dict(size=10, color='white'),
            name=f'S={s}',
            legendgroup=f'S={s}',
            showlegend=True,
            hovertemplate=f"<b>{row['config_label']}</b><br>PPL: {row['wm_ppl_avg']:.2f}<br>TPR: {row['wm_tpr']:.0%}<br>Gap: {row['gap']:.2f}<extra></extra>"
        ),
        row=1, col=1
    )

# Highlight optimal (S=7)
optimal = df_optimal[df_optimal['num_states'] == 7].iloc[0]
fig.add_annotation(
    x=optimal['wm_ppl_avg'],
    y=optimal['wm_tpr'] + 0.08,
    text="🏆 Best",
    showarrow=False,
    font=dict(size=12, color='green'),
    row=1, col=1
)

# Right panel: configurations with overlap
overlap_symbols = {'5%': 'circle', '10%': 'diamond', '15%': 'square'}
for _, row in df_degraded.iterrows():
    s = int(row['num_states'])
    fig.add_trace(
        go.Scatter(
            x=[row['wm_ppl_avg']],
            y=[row['wm_tpr']],
            mode='markers',
            marker=dict(
                size=12 + row['gap']*10,
                color=state_colors[s],
                symbol=overlap_symbols.get(row['overlap_pct'], 'circle'),
                line=dict(width=1, color='white')
            ),
            name=f'S={s}, ρ={row["overlap_pct"]}',
            legendgroup=f'S={s}',
            showlegend=False,
            hovertemplate=f"<b>{row['config_label']}</b><br>PPL: {row['wm_ppl_avg']:.2f}<br>TPR: {row['wm_tpr']:.0%}<br>Gap: {row['gap']:.2f}<extra></extra>"
        ),
        row=1, col=2
    )

# Add overlap legend annotations on right panel
fig.add_annotation(x=7.5, y=0.85, text="○ 5% overlap", showarrow=False, font=dict(size=10), row=1, col=2)
fig.add_annotation(x=7.5, y=0.75, text="◇ 10% overlap", showarrow=False, font=dict(size=10), row=1, col=2)
fig.add_annotation(x=7.5, y=0.65, text="□ 15% overlap", showarrow=False, font=dict(size=10), row=1, col=2)

# Update axes
fig.update_xaxes(title_text='Perplexity (PPL)', row=1, col=1)
fig.update_xaxes(title_text='Perplexity (PPL)', row=1, col=2)
fig.update_yaxes(title_text='Detection Rate (TPR)', tickformat=',.0%', range=[-0.05, 1.1], row=1, col=1)
fig.update_yaxes(tickformat=',.0%', range=[-0.05, 1.1], row=1, col=2)

fig.update_layout(
    title=dict(
        text='<b>Quality-Detection Trade-off</b><br><sup>Bubble size ∝ Score Gap | Ideal: Lower PPL, Higher Detection</sup>',
        x=0.5
    ),
    template='plotly_white',
    height=500,
    width=1100,
    legend=dict(
        title='States',
        orientation='h',
        yanchor='top',
        y=-0.15,
        xanchor='center',
        x=0.5
    ),
    margin=dict(b=100)
)

fig.write_image(f'{PICTURES_DIR}/ppl_detection_tradeoff.png', scale=2)
fig.show()

## 6. Theoretical Validation

Validating theoretical properties: perfect separability and baseline convergence.

In [75]:
# Figure 3: Theoretical Validation (2x1)
from plotly.subplots import make_subplots

df_0pct = df[df['overlap'] == 0.0].copy()
df_sep = df_0pct[df_0pct['num_states'] >= 5].copy()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Perfect Separability: Min WM > Max Non-WM</b>', '<b>Theoretical vs Observed Baseline</b>'),
    horizontal_spacing=0.12
)

# Left: Separability
fig.add_trace(go.Scatter(
    x=df_sep['num_states'].astype(str), y=df_sep['wm_min'], mode='markers+text',
    name='Min WM Score', marker=dict(size=15, color='#2ecc71', symbol='triangle-up'),
    text=df_sep['wm_min'].round(3), textposition='top center'
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df_sep['num_states'].astype(str), y=df_sep['non_wm_max'], mode='markers+text',
    name='Max Non-WM Score', marker=dict(size=15, color='#e74c3c', symbol='triangle-down'),
    text=df_sep['non_wm_max'].round(3), textposition='bottom center'
), row=1, col=1)
fig.add_hline(y=0.5, line_dash="dash", line_color="orange", row=1, col=1)

# Right: Baseline Comparison
fig.add_trace(go.Scatter(
    x=df_0pct['num_states'], y=df_0pct['non_wm_avg'], mode='markers+lines',
    name='Observed Average', marker=dict(size=10, color='#3498db'), line=dict(width=2)
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=df_0pct['num_states'], y=df_0pct['baseline'], mode='markers+lines',
    name='Theoretical (1/S)', marker=dict(size=10, color='#e67e22', symbol='diamond'), line=dict(width=2, dash='dash')
), row=1, col=2)

fig.update_xaxes(title_text='Number of States (S)', row=1, col=1)
fig.update_xaxes(title_text='Number of States (S)', row=1, col=2)
fig.update_yaxes(title_text='Fingerprint Score φ(t)', row=1, col=1)
fig.update_yaxes(title_text='Fingerprint Score', row=1, col=2)

fig.update_layout(
    title=dict(text='<b>Figure 3: Theoretical Properties Validation</b><br><sup>Validates zero-error detection and E[φ(t^r)] = 1/S baseline</sup>', x=0.5),
    template='plotly_white', height=450, width=1100,
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
    margin=dict(b=80)
)

fig.write_image(f'{PICTURES_DIR}/fig3_theoretical.png', scale=2)
fig.show()


## 7. Multi-Metric Configuration Comparison

Radar chart comparing top configurations across multiple metrics.

In [76]:
# Radar chart comparing top configurations
top_configs = df[(df['overlap'] == 0.0) & (df['num_states'].isin([5, 7, 9, 11, 15]))].copy()

# Normalize metrics for radar chart (all 0-1 scale)
top_configs['norm_gap'] = top_configs['gap'] / top_configs['gap'].max()
top_configs['norm_tpr'] = top_configs['wm_tpr']
top_configs['norm_fpr_inv'] = 1 - top_configs['non_wm_fpr']  # Invert so higher is better
top_configs['norm_ppl_inv'] = 1 - (top_configs['wm_ppl_avg'] - top_configs['wm_ppl_avg'].min()) / (top_configs['wm_ppl_avg'].max() - top_configs['wm_ppl_avg'].min())

categories = ['Score Gap', 'Detection Rate', '1 - FPR', 'Quality (1/PPL)']

fig = go.Figure()

colors = px.colors.qualitative.Set1
for i, (_, row) in enumerate(top_configs.iterrows()):
    values = [row['norm_gap'], row['norm_tpr'], row['norm_fpr_inv'], row['norm_ppl_inv']]
    values.append(values[0])  # Close the polygon
    
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=categories + [categories[0]],
        fill='toself',
        name=f"S={int(row['num_states'])}",
        line_color=colors[i % len(colors)],
        opacity=0.7
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1])
    ),
    title=dict(text='<b>Multi-Metric Configuration Comparison</b><br><sup>0% Overlap Configurations | Larger area = Better overall</sup>', x=0.5),
    template='plotly_white',
    height=550,
    width=700,
    legend=dict(title='States', orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5)
)

fig.write_image(f'{PICTURES_DIR}/radar_comparison.png', scale=2)
fig.show()

---

## 8. Summary and Saved Plots

All plots have been saved to `../docs/pictures/` as high-resolution PNG files:

| Figure | Filename | Description |
|--------|----------|-------------|
| Figure 1 | `fig1_score_fpr.png` | Score comparison + FPR analysis |
| Figure 2 | `fig2_overlap_impact.png` | Heatmap + detection degradation |
| Figure 3 | `fig3_theoretical.png` | Separability + baseline validation |
| Figure 4 | `ppl_detection_tradeoff.png` | Quality-detection trade-off |
| Figure 5 | `radar_comparison.png` | Multi-metric radar chart |

In [77]:
# List all generated plots
import os
plots = [f for f in os.listdir(PICTURES_DIR) if f.endswith('.png')]
print(f"Generated {len(plots)} plots:")
for p in sorted(plots):
    print(f"  - {PICTURES_DIR}/{p}")

Generated 5 plots:
  - ../docs/pictures/fig1_score_fpr.png
  - ../docs/pictures/fig2_overlap_impact.png
  - ../docs/pictures/fig3_theoretical.png
  - ../docs/pictures/ppl_detection_tradeoff.png
  - ../docs/pictures/radar_comparison.png
